# LangChain + Enclave Integration Example

This notebook demonstrates how to use `langchain-enclave` to securely access secrets and personalized knowledge in LangChain agents.


## Setup

Install dependencies and configure API keys.


In [ ]:
# Install packages
!pip install langchain-enclave langchain openai


In [ ]:
import os
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain_enclave import EnclaveSecretProvider, EnclaveKnowledgeRetriever

# Configuration
ENCLAVE_API_KEY = os.getenv("ENCLAVE_API_KEY", "vlt_your_key_here")
ENCLAVE_BASE_URL = os.getenv("ENCLAVE_BASE_URL", "https://your-backend.railway.app")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


## Example 1: Secret Retrieval

Create an agent that can retrieve API keys from Enclave.


In [ ]:
# Initialize secret provider
secret_tool = EnclaveSecretProvider(
    api_key=ENCLAVE_API_KEY,
    base_url=ENCLAVE_BASE_URL
)

# Create agent
llm = ChatOpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)
agent = initialize_agent(
    tools=[secret_tool],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

# Use agent
response = agent.run("Get my OpenAI API key")
print(response)


## Example 2: Knowledge Query

Query personalized knowledge via DoRA adapters.


In [ ]:
from langchain.chains import RetrievalQA

# Initialize knowledge retriever
retriever = EnclaveKnowledgeRetriever(
    adapter_id="your-adapter-uuid",
    api_key=ENCLAVE_API_KEY,
    base_url=ENCLAVE_BASE_URL
)

# Create QA chain
qa = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(openai_api_key=OPENAI_API_KEY),
    retriever=retriever
)

# Query knowledge
answer = qa.run("What did I write about project X?")
print(answer)
